# 🌌 Universal Astrological OKF Engine
This notebook serves as a multi-tenant engine for calculating high-precision astrological charts and converting them into the **Open Knowledge Format (OKF)**—a modular, human-readable system designed for knowledge management tools like Obsidian.

In [ ]:
import sys

# Check for Google Colab environment
is_colab = 'google.colab' in sys.modules

missing_packages = []
for pkg_name, import_name in [
    ("pysweph", "swisseph"),
    ("pyyaml", "yaml"),
    ("requests", "requests"),
    ("tzdata", "tzdata")
]:
    try:
        __import__(import_name)
    except ImportError:
        missing_packages.append(pkg_name)

if not missing_packages:
    print("✅ All required packages are installed.")
else:
    print(f"❌ Missing packages: {', '.join(missing_packages)}")
    if is_colab:
        print("Running in Google Colab: Installing missing packages automatically...")
        import subprocess
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
            print("✅ Packages installed successfully. Please restart runtime if needed.")
        except Exception as e:
            print(f"❌ Error installing packages: {e}")
    else:
        print("\nPlease run the following command in your terminal to install them:")
        print(f"    {sys.executable} -m pip install {' '.join(missing_packages)}")
        print("\nAfter installation, please restart the Jupyter kernel in VS Code.")


### 📦 Libraries and Dependencies
This block imports the necessary Python standard libraries and external packages.
- `swisseph` (as `swe`) is the core engine for astronomical calculations.
- `yaml`, `json`, and `hashlib` are used for data formatting and file namespacing.
- `zoneinfo` and `datetime` handle high-precision time conversions.

In [ ]:
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# --- STANDARD LIBRARY IMPORTS ---
import os
import json
import hashlib
import shutil
import types
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo

# --- THIRD-PARTY IMPORTS ---
import requests
import yaml
import swisseph as swe


## 🛰️ Ephemeris Data Acquisition
To ensure scientific-grade precision, we utilize the **Swiss Ephemeris**. This section downloads the necessary planetary data files (`.se1`) from the official repository. These files are required for calculating 'True' planetary positions and node calculations.

In [ ]:
# 1. Create directory
# To run in Google Colab, change this path back to '/content/ephe'
notebook_dir = os.path.abspath('')
ephe_path = os.path.join(notebook_dir, 'ephe')
if not os.path.exists(ephe_path):
    os.makedirs(ephe_path)

# 2. Correct raw URLs
base_url = "https://github.com/aloistr/swisseph/raw/master/ephe/"
urls = [
    base_url + "seas_18.se1",
    base_url + "sepl_18.se1",
    base_url + "semo_18.se1"
]

for url in urls:
    file_name = url.split('/')[-1]
    target = os.path.join(ephe_path, file_name)

    # Cleanup existing corrupt/small files
    if os.path.exists(target) and os.path.getsize(target) < 5000:
        os.remove(target)

    if not os.path.exists(target):
        print(f"Downloading {file_name}...")
        try:
            r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            if r.status_code == 200:
                with open(target, 'wb') as f:
                    f.write(r.content)
                print(f"Successfully downloaded {file_name}")
            else:
                print(f"Failed to download {file_name}: Status {r.status_code}")
        except Exception as e:
            print(f"Error downloading {file_name}: {e}")
    else:
        print(f"✅ {file_name} already exists. Skipping download.")

# 3. Configure and Verify
swe.set_ephe_path(ephe_path)
files_info = {f: f"{os.path.getsize(os.path.join(ephe_path, f)) / 1024:.2f} KB" for f in os.listdir(ephe_path) if f.endswith('.se1')}
print(f"Ephemeris path set to: {ephe_path}")
print(f"Verified Files: {files_info}")


### 🛠️ Configuration and User Inputs
The form below allows you to define the subject's birth data (Date, Time, Location) and specific astrological settings like the **Ayanamsa** (sidereal offset) and **House System**. These variables drive the entire calculation engine.

## ⚙️ Profile Configuration
Use the form below to set the subject's birth data.
- **Subject Name**: Used for namespacing directories and files.
- **Coordinates**: High-precision latitude/longitude is required for accurate House and Ascendant calculations.
- **Settings**: Choose your preferred Ayanamsa (Sidereal Offset) and House System (e.g., Whole Sign or Placidus).

### 🧠 Core Astrological Engine
This section defines the mathematical and logical framework for the chart:
- `AstroTime`: Converts local time to Julian Day (JD) and handles UTC offsets.
- `AstroMath`: Provides utility functions for mapping longitudes to signs and nakshatras.
- `VedicAspectEngine`: Logic for calculating planetary sight (Graha Drishti).
- `FriendshipEngine`: Calculates natural and temporary relationships between planets (Panchadha Maitri).
- `DignityEngine`: Determines planetary strength (Exaltation, Moolatrikona, etc.).
- `VimshottariDashaEngine`: Projects time-based planetary cycles based on the Moon's position.

In [ ]:
# @title ⚙️ Birth Data Configuration

SUBJECT_NAME = "Steve Jobs" # @param {type:"string"}
B_YEAR = 1955 # @param {type:"integer"}
B_MONTH = 2 # @param {type:"integer"}
B_DAY = 24 # @param {type:"integer"}
B_TIME = "19:15" # @param {type:"string"}
B_TZ = "America/Los_Angeles" # @param {type:"string"}
GENDER = "Male" # @param ["Male", "Female"]

LATITUDE = 37.7749 # @param {type:"number"}
LONGITUDE = -122.4194 # @param {type:"number"}

# @markdown ---
# @markdown ### Astrological Settings
# @markdown **Report Mode:** Controls what data is included in the OKF output.
# @markdown - `Tropical-Vedic`: Tropical Signs/Houses + Vedic Nakshatras/Dashas/Aspects.
REPORT_MODE = "Tropical-Vedic" # @param ["Tropical Focus", "Vedic Focus", "Everything", "Tropical-Vedic"]

HOUSE_SYSTEM = "W" # @param ["W", "P", "E"]
AYANAMSA = 36 # @param [36, 1, 3, 5]

# --- AUTO-COMPUTED EXPORT PATHS & METADATA ---
import os
import hashlib

safe_subject = SUBJECT_NAME.lower().replace(" ", "_")
target_json_path = os.path.join("knowledge_base", safe_subject, f"{safe_subject}_astrology_profile.json")

# Generate unique fingerprint based on input parameters
fingerprint_input = f"{SUBJECT_NAME}_{B_YEAR}_{B_MONTH}_{B_DAY}_{B_TIME}_{LATITUDE}_{LONGITUDE}_{HOUSE_SYSTEM}_{AYANAMSA}"
fingerprint = hashlib.md5(fingerprint_input.encode('utf-8')).hexdigest()[:8]

subject_profile = {
    "name": SUBJECT_NAME,
    "gender": GENDER,
    "birth_data": {
        "date": f"{B_YEAR}-{B_MONTH:02d}-{B_DAY:02d}",
        "time": B_TIME,
        "timezone": B_TZ,
        "latitude": LATITUDE,
        "longitude": LONGITUDE
    }
}

metadata = {
    "engine": "Universal Astrological OKF Engine",
    "version": "2.0",
    "settings": {
        "report_mode": REPORT_MODE,
        "house_system": HOUSE_SYSTEM,
        "house_system_name": "Placidus" if HOUSE_SYSTEM == "P" else ("Equal" if HOUSE_SYSTEM == "E" else "Whole Sign"),
        "ayanamsa": AYANAMSA,
        "ayanamsa_name": "Galactic Center at 0 Mula (Wilhelm)" if AYANAMSA == 36 else (
            "Lahiri" if AYANAMSA == 1 else (
                "Raman" if AYANAMSA == 3 else (
                    "Fagan-Bradley" if AYANAMSA == 0 else (
                        "KP" if AYANAMSA == 5 else f"Custom ({AYANAMSA})"
                    )
                )
            )
        )
    }
}


# 🔱 Astrological Calculation Engines & Exporter

This section defines the unified calculation engines for computing divisional harmonics, planetary strength arrays, auspicious point distributions, and planetary yoga configurations. All engines dynamically adapt to the active `REPORT_MODE` (Tropical vs. Vedic coordinate frames).

### ⚙️ Engine Summary
1. **`AstroMath`**: Core utilities for sign, nakshatra, and house placements.
2. **`VargaEngine`**: Computes harmonic divisions (D1 through D60) based on active longitudes.
3. **`ShadbalaEngine`**: Calculates planetary strength across 6 categories (Sthana, Dig, Kala, Cheshta, Naisargika, Drik).
4. **`AshtakavargaEngine`**: Computes Bhinna (BAV) and Sarvashtakavarga (SAV) point matrices.
5. **`YogaEngine`**: Scans the birth chart for classic planetary yogas.
6. **`GocharaEngine`**: Evaluates transit alignments relative to the natal Moon position.
7. **`OKFConverter`**: Outputs structured, Obsidian-linked Markdown reports.

In [ ]:
import os
import json
import yaml
import hashlib
import shutil
import types
from datetime import datetime, timezone, timedelta
import swisseph as swe

# --- 1. CORE TIME & COORDINATE ENGINES (SELF-CONTAINED) ---
class AstroTime:
    def __init__(self, year, month, day, hour: float | str, tz: float | str, sid_mode: str | int = "Lahiri"):
        # Local time to UTC conversion
        self.year = year
        self.month = month
        self.day = day
        self.hour = hour
        self.tz = tz
        self.sid_mode = sid_mode
        
        # hour is a string in format "HH:MM" (e.g. "1:18") or a float/int
        if isinstance(hour, str):
            parts = hour.split(':')
            h = int(parts[0])
            m = int(parts[1]) if len(parts) > 1 else 0
            s = int(parts[2]) if len(parts) > 2 else 0
        else:
            dec_hour_local = float(hour)
            h = int(dec_hour_local)
            m = int((dec_hour_local - h) * 60)
            s = int(round((dec_hour_local - h - m/60.0) * 3600))

        # Handle minute/second overflow
        if s >= 60:
            m += s // 60
            s %= 60
        if m >= 60:
            h += m // 60
            m %= 60

        if isinstance(tz, str):
            from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
            dt_naive = datetime(year, month, day, h, m, s)
            try:
                tz_obj = ZoneInfo(tz)
            except ZoneInfoNotFoundError:
                raise ZoneInfoNotFoundError(
                    f"Timezone '{tz}' was not found in the timezone database. "
                    "If you are running in Google Colab, you must install the 'tzdata' package. "
                    "You can do this by running `!pip install tzdata` in a code cell and then restarting the runtime."
                )
            dt_local = dt_naive.replace(tzinfo=tz_obj)
            dt_utc = dt_local.astimezone(timezone.utc)
            dec_hour_utc = dt_utc.hour + dt_utc.minute/60.0 + dt_utc.second/3600.0
            self.jd = swe.julday(dt_utc.year, dt_utc.month, dt_utc.day, dec_hour_utc)
        else:
            dt_local = datetime(year, month, day, h, m, s)
            dt_utc = dt_local - timedelta(hours=tz)
            dec_hour_utc = dt_utc.hour + dt_utc.minute/60.0 + dt_utc.second/3600.0
            self.jd = swe.julday(dt_utc.year, dt_utc.month, dt_utc.day, dec_hour_utc)
        
        # Configure Swiss Ephemeris Ayanamsa mode
        ayanamsas = {
            "Lahiri": swe.SIDM_LAHIRI,
            "Raman": swe.SIDM_RAMAN,
            "Fagan-Bradley": swe.SIDM_FAGAN_BRADLEY,
            "Kp": swe.SIDM_KRISHNAMURTI,
            "Usha-Shashi": swe.SIDM_USHASHASHI,
            "Yukteshwar": swe.SIDM_YUKTESHWAR,
            "Jn-Bhasin": swe.SIDM_JN_BHASIN
        }
        if isinstance(sid_mode, int):
            mode_const = sid_mode
        else:
            mode_const = ayanamsas.get(sid_mode, swe.SIDM_LAHIRI)
        swe.set_sid_mode(mode_const)

class HouseEngine:
    def __init__(self, astro_time, lat, lon, house_system="W"):
        self.jd = astro_time.jd
        self.lat = lat
        self.lon = lon
        self.house_system = house_system
        if isinstance(house_system, str):
            hsys = house_system.encode('ascii')
        elif isinstance(house_system, int):
            hsys = bytes([house_system])
        else:
            hsys = b'P'
        
        # Calculate Tropical houses
        cusps_t, asmc_t = swe.houses_ex(self.jd, lat, lon, hsys)
        self.tropical_ascendant = asmc_t[0]
        self.tropical_ascendant_sign = AstroMath.get_zodiac(self.tropical_ascendant)
        
        # Calculate Sidereal houses
        cusps_s, asmc_s = swe.houses_ex(self.jd, lat, lon, hsys, flags=swe.FLG_SIDEREAL)
        self.sidereal_ascendant = asmc_s[0]
        self.sidereal_ascendant_sign = AstroMath.get_zodiac(self.sidereal_ascendant)
        
        # Handle 12-item vs 13-item cusp tuples from different pyswisseph/pysweph versions
        t_offset = 0 if len(cusps_t) == 13 else -1
        s_offset = 0 if len(cusps_s) == 13 else -1
        
        self.houses = {}
        for h in range(1, 13):
            self.houses[h] = {
                "tropical_start": cusps_t[h + t_offset],
                "sidereal_start": cusps_s[h + s_offset]
            }
# --- 2. ASTRO MATH & DIGNITY UTILITIES ---
class AstroMath:
    ZODIAC_SIGNS = ["Aries", "Taurus", "Gemini", "Cancer", "Leo", "Virgo", "Libra", "Scorpio", "Sagittarius", "Capricorn", "Aquarius", "Pisces"]
    NAKSHATRAS = ["Ashwini", "Bharani", "Krittika", "Rohini", "Mrigashira", "Ardra", "Punarvasu", "Pushya", "Ashlesha", "Magha", "Purva Phalguni", "Uttara Phalguni", "Hasta", "Chitra", "Swati", "Vishakha", "Anuradha", "Jyeshtha", "Mula", "Purva Ashadha", "Uttara Ashadha", "Shravana", "Dhanishta", "Shatabhisha", "Purva Bhadrapada", "Uttara Bhadrapada", "Revati"]

    @classmethod
    def get_zodiac(cls, longitude):
        return cls.ZODIAC_SIGNS[int(longitude // 30) % 12]

    @classmethod
    def get_nakshatra(cls, longitude):
        return cls.NAKSHATRAS[int(longitude / 13.333333333333334) % 27]

    @classmethod
    def get_house_placement(cls, longitude, houses_dict, is_sidereal=False):
        prefix = "sidereal_start" if is_sidereal else "tropical_start"
        for h in range(1, 13):
            nxt = h + 1 if h < 12 else 1
            s, e = houses_dict[h][prefix], houses_dict[nxt][prefix]
            if (s < e and s <= longitude < e) or (s > e and (longitude >= s or longitude < e)):
                return h
        return 1

    @classmethod
    def get_degree_in_house(cls, longitude, house_num, houses_dict, is_sidereal=False):
        prefix = "sidereal_start" if is_sidereal else "tropical_start"
        cusp = houses_dict[house_num][prefix]
        diff = (longitude - cusp) % 360
        return diff

class DignityEngine:
    EXALTATION_SIGNS = {
        "Sun": "Aries", "Moon": "Taurus", "Mars": "Capricorn", "Mercury": "Virgo",
        "Jupiter": "Cancer", "Venus": "Pisces", "Saturn": "Libra"
    }
    DEBILITATION_SIGNS = {
        "Sun": "Libra", "Moon": "Scorpio", "Mars": "Cancer", "Mercury": "Pisces",
        "Jupiter": "Capricorn", "Venus": "Virgo", "Saturn": "Aries"
    }
    OWN_SIGNS = {
        "Sun": ["Leo"],
        "Moon": ["Cancer"],
        "Mars": ["Aries", "Scorpio"],
        "Mercury": ["Gemini", "Virgo"],
        "Jupiter": ["Sagittarius", "Pisces"],
        "Venus": ["Taurus", "Libra"],
        "Saturn": ["Capricorn", "Aquarius"]
    }
    
    @classmethod
    def get_dignity(cls, name, sign):
        if name not in cls.OWN_SIGNS:
            return "Neutral"
        if cls.EXALTATION_SIGNS.get(name) == sign:
            return "Exalted"
        if cls.DEBILITATION_SIGNS.get(name) == sign:
            return "Debilitated"
        if sign in cls.OWN_SIGNS.get(name, []):
            return "Own Sign"
        return "Neutral"

class FriendshipEngine:
    NATURAL = {
        "Sun": {"friends": ["Moon", "Mars", "Jupiter"], "enemies": ["Venus", "Saturn"], "neutrals": ["Mercury"]},
        "Moon": {"friends": ["Sun", "Mercury"], "enemies": [], "neutrals": ["Mars", "Jupiter", "Venus", "Saturn"]},
        "Mars": {"friends": ["Sun", "Moon", "Jupiter"], "enemies": ["Mercury"], "neutrals": ["Venus", "Saturn"]},
        "Mercury": {"friends": ["Sun", "Venus"], "enemies": ["Moon"], "neutrals": ["Mars", "Jupiter", "Saturn"]},
        "Jupiter": {"friends": ["Sun", "Moon", "Mars"], "enemies": ["Mercury", "Venus"], "neutrals": ["Saturn"]},
        "Venus": {"friends": ["Mercury", "Saturn"], "enemies": ["Sun", "Moon"], "neutrals": ["Mars", "Jupiter"]},
        "Saturn": {"friends": ["Mercury", "Venus"], "enemies": ["Sun", "Moon", "Mars"], "neutrals": ["Jupiter"]}
    }

    @classmethod
    def get_panchadha_relationship(cls, p1, p2, h1, h2):
        if p1 not in cls.NATURAL or p2 not in cls.NATURAL:
            return "Sama"
            
        p1_nat = cls.NATURAL[p1]
        if p2 in p1_nat["friends"]:
            nat_score = 1
        elif p2 in p1_nat["enemies"]:
            nat_score = -1
        else:
            nat_score = 0
            
        diff = (h2 - h1) % 12
        if diff in [1, 2, 3, 9, 10, 11]:
            temp_score = 1
        else:
            temp_score = -1
            
        total = nat_score + temp_score
        mapping = {2: "Adhi Mitra", 1: "Mitra", 0: "Sama", -1: "Shatru", -2: "Adhi Shatru"}
        return mapping.get(total, "Sama")

# --- 3. VARGA & PLANET DEFINITIONS ---
class VargaEngine:
    @staticmethod
    def get_vargas(planet_obj, mode):
        if mode == "Tropical Focus":
            return {}
        lon = planet_obj.tropical_longitude if mode == "Tropical-Vedic" else planet_obj.sidereal_longitude
        return {
            "D1": VargaEngine.calculate_d1(lon),
            "D2": VargaEngine.calculate_d2(lon),
            "D3": VargaEngine.calculate_d3(lon),
            "D7": VargaEngine.calculate_d7(lon),
            "D9": VargaEngine.calculate_d9(lon),
            "D10": VargaEngine.calculate_d10(lon),
            "D12": VargaEngine.calculate_d12(lon),
            "D60": VargaEngine.calculate_d60(lon)
        }

    @staticmethod
    def calculate_d1(lon):
        return AstroMath.ZODIAC_SIGNS[int(lon // 30) % 12]

    @staticmethod
    def calculate_d2(lon):
        sign_idx = int(lon // 30) % 12
        offset = lon % 30
        if sign_idx % 2 == 0:  # Odd sign
            return "Leo" if offset < 15 else "Cancer"
        else:  # Even sign
            return "Cancer" if offset < 15 else "Leo"

    @staticmethod
    def calculate_d3(lon):
        sign_idx = int(lon // 30) % 12
        part = int((lon % 30) // 10)
        return AstroMath.ZODIAC_SIGNS[(sign_idx + part * 4) % 12]

    @staticmethod
    def calculate_d7(lon):
        sign_idx = int(lon // 30) % 12
        part = int((lon % 30) // (30 / 7))
        start = sign_idx if sign_idx % 2 == 0 else (sign_idx + 6) % 12
        return AstroMath.ZODIAC_SIGNS[(start + part) % 12]

    @staticmethod
    def calculate_d9(lon):
        return AstroMath.ZODIAC_SIGNS[int(lon / (3 + 1/3)) % 12]

    @staticmethod
    def calculate_d10(lon):
        s_idx, p_idx = int(lon // 30), int((lon % 30) // 3)
        start = s_idx if s_idx % 2 == 0 else (s_idx + 8) % 12
        return AstroMath.ZODIAC_SIGNS[(start + p_idx) % 12]

    @staticmethod
    def calculate_d12(lon):
        return AstroMath.ZODIAC_SIGNS[(int(lon // 30) + int((lon % 30) // 2.5)) % 12]

    @staticmethod
    def calculate_d60(lon):
        return AstroMath.ZODIAC_SIGNS[(int(lon // 30) + int((lon % 30) // 0.5)) % 12]

class Planet:
    def __init__(self, name, swe_const, astro_time, mode="Everything", base_planet=None):
        self.name, self.jd = name, astro_time.jd
        self.tropical_house: int | None = None
        self.vedic_house: int | None = None
        self.shadbala: dict | None = None
        if base_planet:
            self.tropical_longitude = (base_planet.tropical_longitude + 180) % 360
            self.tropical_speed = base_planet.tropical_speed
            self.sidereal_longitude = (base_planet.sidereal_longitude + 180) % 360
            self.sidereal_speed = base_planet.sidereal_speed
        else:
            t_d = swe.calc_ut(self.jd, swe_const, swe.FLG_SPEED)[0]
            self.tropical_longitude, self.tropical_speed = t_d[0], t_d[3]
            s_d = swe.calc_ut(self.jd, swe_const, swe.FLG_SIDEREAL | swe.FLG_SPEED)[0]
            self.sidereal_longitude, self.sidereal_speed = s_d[0], s_d[3]
            
        self.zodiac_sign = AstroMath.get_zodiac(self.tropical_longitude)
        self.sidereal_sign = AstroMath.get_zodiac(self.sidereal_longitude)
        self.nakshatra = AstroMath.get_nakshatra(self.sidereal_longitude)
        self.tropical_dignity = DignityEngine.get_dignity(self.name, self.zodiac_sign)
        self.sidereal_dignity = DignityEngine.get_dignity(self.name, self.sidereal_sign)
        self.dignity = self.sidereal_dignity if mode in ["Vedic Focus", "Everything"] else self.tropical_dignity
        self.vargas = VargaEngine.get_vargas(self, mode)

class ChartFactory:
    PLANETS = {"Sun": swe.SUN, "Moon": swe.MOON, "Mercury": swe.MERCURY, "Venus": swe.VENUS, "Mars": swe.MARS, "Jupiter": swe.JUPITER, "Saturn": swe.SATURN, "Rahu": swe.TRUE_NODE}
    @classmethod
    def create_chart(cls, astro_time, mode="Everything"):
        chart = {n: Planet(n, c, astro_time, mode) for n, c in cls.PLANETS.items()}
        rahu = chart["Rahu"]
        ketu = Planet("Ketu", None, astro_time, mode, base_planet=rahu)
        chart["Ketu"] = ketu
        return chart

# --- 4. SHADBALA CALCULATIONS ---
class ShadbalaEngine:
    EXALTATIONS = {
        "Sun": 10.0, "Moon": 33.0, "Mars": 298.0, "Mercury": 165.0,
        "Jupiter": 95.0, "Venus": 357.0, "Saturn": 200.0
    }
    LORDSHIPS = {
        "Aries": "Mars", "Scorpio": "Mars",
        "Taurus": "Venus", "Libra": "Venus",
        "Gemini": "Mercury", "Virgo": "Mercury",
        "Cancer": "Moon", "Leo": "Sun",
        "Sagittarius": "Jupiter", "Pisces": "Jupiter",
        "Capricorn": "Saturn", "Aquarius": "Saturn"
    }

    @classmethod
    def calculate_ucca_bala(cls, planet_name, longitude):
        if planet_name not in cls.EXALTATIONS:
            return 0.0
        exalt = cls.EXALTATIONS[planet_name]
        debilitation = (exalt + 180) % 360
        dist = abs(longitude - debilitation) % 360
        if dist > 180:
            dist = 360 - dist
        return round(dist / 3.0, 4)

    @classmethod
    def calculate_kendra_bala(cls, house_num):
        if house_num in [1, 4, 7, 10]:
            return 60.0
        elif house_num in [2, 5, 8, 11]:
            return 30.0
        else:
            return 15.0

    @classmethod
    def calculate_saptavargiya_bala(cls, planet_name, planet_obj, chart, houses_dict, mode):
        if planet_name not in cls.EXALTATIONS:
            return 0.0
            
        total_points = 0.0
        vargas = planet_obj.vargas
        varga_list = ["D1", "D2", "D3", "D7", "D9", "D10", "D12"]
        
        for v in varga_list:
            varga_sign = vargas.get(v)
            if not varga_sign:
                continue
            lord = cls.LORDSHIPS.get(varga_sign)
            
            if lord == planet_name:
                total_points += 30.0
            else:
                is_sid = False if mode in ["Tropical-Vedic", "Tropical Focus"] else True
                h_planet = AstroMath.get_house_placement(planet_obj.sidereal_longitude if is_sid else planet_obj.tropical_longitude, houses_dict, is_sid)
                h_lord = AstroMath.get_house_placement(chart[lord].sidereal_longitude if is_sid else chart[lord].tropical_longitude, houses_dict, is_sid)
                rel = FriendshipEngine.get_panchadha_relationship(planet_name, lord, h_planet, h_lord)
                
                relationship_scores = {
                    "Adhi Mitra": 22.5, "Mitra": 15.0, "Sama": 10.0, "Shatru": 4.0, "Adhi Shatru": 2.0
                }
                total_points += relationship_scores.get(rel, 10.0)
                
        return round(total_points, 4)

    @classmethod
    def get_sthana_bala(cls, planet_name, planet_obj, chart, houses_dict, house_num, mode):
        if planet_name not in cls.EXALTATIONS:
            return {}
        is_sid = False if mode in ["Tropical-Vedic", "Tropical Focus"] else True
        lon = planet_obj.sidereal_longitude if is_sid else planet_obj.tropical_longitude
        
        ucca = cls.calculate_ucca_bala(planet_name, lon)
        kendra = cls.calculate_kendra_bala(house_num)
        saptavargiya = cls.calculate_saptavargiya_bala(planet_name, planet_obj, chart, houses_dict, mode)
        total = ucca + kendra + saptavargiya
        return {
            "ucca_bala": ucca,
            "kendra_bala": kendra,
            "saptavargiya_bala": saptavargiya,
            "sthana_bala_total": round(total, 4)
        }

    @classmethod
    def calculate_dig_bala(cls, planet_name, house_num):
        max_houses = {
            "Jupiter": 1, "Mercury": 1,
            "Sun": 10, "Mars": 10,
            "Saturn": 7,
            "Moon": 4, "Venus": 4
        }
        if planet_name not in max_houses:
            return 0.0
        max_h = max_houses[planet_name]
        d = min((house_num - max_h) % 12, (max_h - house_num) % 12)
        return float((6 - d) * 10)

    @classmethod
    def calculate_divaratri_bala(cls, planet_name, is_day):
        day_planets = ["Sun", "Jupiter", "Venus"]
        night_planets = ["Moon", "Mars", "Saturn"]
        if planet_name == "Mercury":
            return 60.0
        elif is_day and planet_name in day_planets:
            return 60.0
        elif not is_day and planet_name in night_planets:
            return 60.0
        return 0.0

    @classmethod
    def calculate_paksha_bala(cls, planet_name, moon_lon, sun_lon):
        if planet_name not in cls.EXALTATIONS:
            return 0.0
        diff = (moon_lon - sun_lon) % 360
        dist = 180.0 - abs(180.0 - diff)
        benefics = ["Jupiter", "Venus", "Moon", "Mercury"]
        if planet_name in benefics:
            return round(dist / 3.0, 4)
        else:
            return round(60.0 - (dist / 3.0), 4)

    @classmethod
    def calculate_cheshta_bala(cls, planet_name, speed):
        if planet_name not in cls.EXALTATIONS:
            return 0.0
        if planet_name in ["Sun", "Moon"]:
            return 30.0
        if speed < 0.0:
            return 60.0
        max_speeds = {"Mars": 1.2, "Mercury": 2.2, "Jupiter": 0.2, "Venus": 1.8, "Saturn": 0.1}
        max_s = max_speeds.get(planet_name, 1.0)
        return round(60.0 * (1.0 - min(abs(speed) / max_s, 1.0)), 4)

    @classmethod
    def calculate_naisargika_bala(cls, planet_name):
        constants = {"Sun": 60.0, "Moon": 51.43, "Venus": 42.86, "Jupiter": 34.29, "Mercury": 25.71, "Mars": 17.14, "Saturn": 8.57}
        return constants.get(planet_name, 0.0)

    @classmethod
    def calculate_drik_bala(cls, planet_name, planet_obj, chart, mode):
        if planet_name not in cls.EXALTATIONS:
            return 0.0
        is_sid = False if mode in ["Tropical-Vedic", "Tropical Focus"] else True
        lon1 = planet_obj.sidereal_longitude if is_sid else planet_obj.tropical_longitude
        drik_score = 0.0
        for name, other in chart.items():
            if name == planet_name or name not in cls.EXALTATIONS:
                continue
            lon2 = other.sidereal_longitude if is_sid else other.tropical_longitude
            diff = abs(lon1 - lon2) % 360
            aspects = [0.0, 60.0, 90.0, 120.0, 180.0]
            has_aspect = False
            for asp in aspects:
                if abs(diff - asp) <= 6.0 or abs(diff - (360.0 - asp)) <= 6.0:
                    has_aspect = True
                    break
            if has_aspect:
                if name in ["Jupiter"]:
                    drik_score += 15.0
                elif name in ["Venus"]:
                    drik_score += 10.0
                elif name in ["Saturn", "Mars"]:
                    drik_score -= 10.0
        return round(drik_score, 4)

    @classmethod
    def get_complete_shadbala(cls, planet_name, planet_obj, chart, houses_dict, house_num, mode):
        sthana = cls.get_sthana_bala(planet_name, planet_obj, chart, houses_dict, house_num, mode)
        if not sthana:
            return {}
        is_sid = False if mode in ["Tropical-Vedic", "Tropical Focus"] else True
        dig = cls.calculate_dig_bala(planet_name, house_num)
        
        sun_house = chart["Sun"].vedic_house if is_sid else chart["Sun"].tropical_house
        is_day = sun_house in [7, 8, 9, 10, 11, 12]
        divaratri = cls.calculate_divaratri_bala(planet_name, is_day)
        
        moon_lon = chart["Moon"].sidereal_longitude if is_sid else chart["Moon"].tropical_longitude
        sun_lon = chart["Sun"].sidereal_longitude if is_sid else chart["Sun"].tropical_longitude
        paksha = cls.calculate_paksha_bala(planet_name, moon_lon, sun_lon)
        kala = round(divaratri + paksha, 4)
        
        speed = planet_obj.sidereal_speed if is_sid else planet_obj.tropical_speed
        cheshta = cls.calculate_cheshta_bala(planet_name, speed)
        naisargika = cls.calculate_naisargika_bala(planet_name)
        drik = cls.calculate_drik_bala(planet_name, planet_obj, chart, mode)
        total = sthana["sthana_bala_total"] + dig + kala + cheshta + naisargika + drik
        return {
            "sthana_bala": sthana,
            "dig_bala": dig,
            "kala_bala": {
                "divaratri_bala": divaratri,
                "paksha_bala": paksha,
                "kala_bala_total": kala
            },
            "cheshta_bala": cheshta,
            "naisargika_bala": naisargika,
            "drik_bala": drik,
            "shadbala_total": round(total, 4)
        }

# --- 5. ASHTAKAVARGA ENGINE ---
class AshtakavargaEngine:
    RULES = {
        "Sun": {
            "Sun": [1, 2, 4, 7, 8, 9, 10, 11],
            "Moon": [3, 6, 10, 11],
            "Mars": [1, 2, 4, 7, 8, 9, 10, 11],
            "Mercury": [5, 6, 9, 11, 12],
            "Jupiter": [5, 6, 9, 11],
            "Venus": [6, 7, 12],
            "Saturn": [1, 2, 4, 7, 8, 9, 10, 11],
            "Lagna": [3, 4, 6, 10, 11, 12]
        },
        "Moon": {
            "Sun": [3, 6, 7, 8, 10, 11],
            "Moon": [1, 3, 6, 7, 10, 11],
            "Mars": [2, 3, 5, 6, 9, 10, 11],
            "Mercury": [1, 3, 4, 5, 7, 8, 10, 11],
            "Jupiter": [1, 4, 7, 8, 10, 11, 12],
            "Venus": [3, 4, 5, 7, 9, 10, 11],
            "Saturn": [3, 5, 6, 11],
            "Lagna": [3, 6, 10, 11]
        },
        "Mars": {
            "Sun": [3, 5, 6, 10, 11],
            "Moon": [3, 6, 11],
            "Mars": [1, 2, 4, 7, 8, 10, 11],
            "Mercury": [3, 5, 6, 11],
            "Jupiter": [6, 10, 11, 12],
            "Venus": [6, 8, 11, 12],
            "Saturn": [1, 4, 7, 8, 9, 10, 11],
            "Lagna": [1, 3, 6, 10, 11]
        },
        "Mercury": {
            "Sun": [5, 6, 9, 11, 12],
            "Moon": [2, 4, 6, 8, 10, 11],
            "Mars": [1, 2, 4, 7, 8, 9, 10, 11],
            "Mercury": [1, 3, 5, 6, 9, 10, 11, 12],
            "Jupiter": [6, 8, 11, 12],
            "Venus": [1, 2, 3, 4, 5, 8, 9, 11],
            "Saturn": [1, 2, 4, 7, 8, 9, 10, 11],
            "Lagna": [1, 2, 4, 6, 8, 10, 11]
        },
        "Jupiter": {
            "Sun": [1, 2, 3, 4, 7, 8, 9, 10, 11],
            "Moon": [2, 5, 7, 9, 11],
            "Mars": [1, 2, 4, 7, 8, 10, 11],
            "Mercury": [1, 2, 4, 5, 6, 9, 10, 11],
            "Jupiter": [1, 2, 3, 4, 7, 8, 10, 11],
            "Venus": [2, 5, 6, 9, 10, 11],
            "Saturn": [3, 5, 6, 12],
            "Lagna": [1, 2, 4, 5, 6, 7, 9, 10, 11]
        },
        "Venus": {
            "Sun": [8, 11, 12],
            "Moon": [1, 2, 3, 4, 5, 8, 9, 11, 12],
            "Mars": [3, 5, 6, 9, 11, 12],
            "Mercury": [3, 5, 6, 9, 11],
            "Jupiter": [5, 8, 9, 10, 11],
            "Venus": [1, 2, 3, 4, 5, 8, 9, 10, 11],
            "Saturn": [3, 4, 5, 8, 9, 10, 11],
            "Lagna": [1, 2, 3, 4, 5, 8, 9, 11]
        },
        "Saturn": {
            "Sun": [1, 2, 4, 7, 8, 10, 11],
            "Moon": [3, 6, 11],
            "Mars": [3, 5, 6, 10, 11, 12],
            "Mercury": [6, 8, 9, 10, 11, 12],
            "Jupiter": [5, 6, 11, 12],
            "Venus": [6, 11, 12],
            "Saturn": [3, 5, 6, 11],
            "Lagna": [1, 3, 4, 6, 10, 11]
        }
    }

    @classmethod
    def calculate_ashtakavarga(cls, chart, local_sky, mode):
        planet_signs = {}
        for name in ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn"]:
            lon = chart[name].tropical_longitude if mode in ["Tropical-Vedic", "Tropical Focus"] else chart[name].sidereal_longitude
            sign_idx = int(lon // 30) % 12
            planet_signs[name] = sign_idx
            
        lagna_sign_name = local_sky.tropical_ascendant_sign if mode in ["Tropical-Vedic", "Tropical Focus"] else local_sky.sidereal_ascendant_sign
        lagna_sign_idx = AstroMath.ZODIAC_SIGNS.index(lagna_sign_name)
        planet_signs["Lagna"] = lagna_sign_idx
        
        bav = {}
        for p_name in ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn"]:
            bav[p_name] = [0] * 12
            p_rules = cls.RULES[p_name]
            
            for sign in range(12):
                points = 0
                for ref_name, auspicious_houses in p_rules.items():
                    ref_sign_idx = planet_signs[ref_name]
                    d = (sign - ref_sign_idx) % 12 + 1
                    if d in auspicious_houses:
                        points += 1
                bav[p_name][sign] = points
                
        sav = [0] * 12
        for sign in range(12):
            sav[sign] = sum(bav[p_name][sign] for p_name in ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn"])
            
        return {
            "bav": bav,
            "sav": sav
        }

# --- 6. YOGA ENGINE ---
class YogaEngine:
    EXALTATION_SIGNS = {
        "Sun": "Aries", "Moon": "Taurus", "Mars": "Capricorn", "Mercury": "Virgo",
        "Jupiter": "Cancer", "Venus": "Pisces", "Saturn": "Libra"
    }
    DEBILITATION_SIGNS = {
        "Sun": "Libra", "Moon": "Scorpio", "Mars": "Cancer", "Mercury": "Pisces",
        "Jupiter": "Capricorn", "Venus": "Virgo", "Saturn": "Aries"
    }
    OWN_SIGNS = {
        "Sun": ["Leo"],
        "Moon": ["Cancer"],
        "Mars": ["Aries", "Scorpio"],
        "Mercury": ["Gemini", "Virgo"],
        "Jupiter": ["Sagittarius", "Pisces"],
        "Venus": ["Taurus", "Libra"],
        "Saturn": ["Capricorn", "Aquarius"]
    }
    LORDSHIPS = {
        "Aries": "Mars", "Scorpio": "Mars",
        "Taurus": "Venus", "Libra": "Venus",
        "Gemini": "Mercury", "Virgo": "Mercury",
        "Cancer": "Moon", "Leo": "Sun",
        "Sagittarius": "Jupiter", "Pisces": "Jupiter",
        "Capricorn": "Saturn", "Aquarius": "Saturn"
    }

    @classmethod
    def get_rules_dignity(cls, name, lon, chart=None, is_sid=True):
        sign = AstroMath.get_zodiac(lon)
        deg = lon % 30
        
        if cls.EXALTATION_SIGNS.get(name) == sign:
            return "Exaltation"
            
        mulatrikona_ranges = {
            "Sun": ("Leo", 0, 20),
            "Moon": ("Taurus", 3, 30),
            "Mars": ("Aries", 0, 12),
            "Mercury": ("Virgo", 15, 20),
            "Jupiter": ("Sagittarius", 0, 10),
            "Venus": ("Libra", 0, 15),
            "Saturn": ("Aquarius", 0, 20)
        }
        if name in mulatrikona_ranges:
            m_sign, start, end = mulatrikona_ranges[name]
            if sign == m_sign and start <= deg < end:
                return "Mulatrikona"
                
        if sign in cls.OWN_SIGNS.get(name, []):
            return "Own"
            
        if cls.DEBILITATION_SIGNS.get(name) == sign:
            return "Debilitated"

        if chart and name in FriendshipEngine.NATURAL:
            lord = cls.LORDSHIPS.get(sign)
            if lord and lord in FriendshipEngine.NATURAL:
                h_planet = chart[name].vedic_house if is_sid else chart[name].tropical_house
                h_lord = chart[lord].vedic_house if is_sid else chart[lord].tropical_house
                rel = FriendshipEngine.get_panchadha_relationship(name, lord, h_planet, h_lord)
                if rel in ["Mitra", "Adhi Mitra"]:
                    return "Friendly"
            
        return "Neutral"

    @classmethod
    def is_unblemished(cls, planet_obj, sun_obj, is_sid):
        name = planet_obj.name
        lon = planet_obj.sidereal_longitude if is_sid else planet_obj.tropical_longitude
        sign = AstroMath.get_zodiac(lon)
        
        if cls.DEBILITATION_SIGNS.get(name) == sign:
            return False
            
        combustion_limits = {
            "Mars": 17.0,
            "Mercury": 14.0,
            "Jupiter": 11.0,
            "Venus": 10.0,
            "Saturn": 15.0
        }
        if name in combustion_limits:
            sun_lon = sun_obj.sidereal_longitude if is_sid else sun_obj.tropical_longitude
            diff = abs(lon - sun_lon) % 360
            if diff > 180:
                diff = 360 - diff
            if diff <= combustion_limits[name]:
                return False
                
        return True

    @classmethod
    def get_sign_quality(cls, lon):
        sign = AstroMath.get_zodiac(lon)
        moveable = ["Aries", "Cancer", "Libra", "Capricorn"]
        fixed = ["Taurus", "Leo", "Scorpio", "Aquarius"]
        if sign in moveable:
            return "Moveable"
        elif sign in fixed:
            return "Fixed"
        return "Dual"

    @classmethod
    def is_planet_strong(cls, planet_name, planet_obj):
        if not planet_obj.shadbala:
            return False
        total = planet_obj.shadbala.get("shadbala_total", 0)
        thresholds = {
            "Sun": 390.0, "Moon": 360.0, "Mars": 300.0, "Mercury": 420.0,
            "Jupiter": 390.0, "Venus": 330.0, "Saturn": 300.0
        }
        return total >= thresholds.get(planet_name, 300.0)

    @classmethod
    def get_ruled_houses(cls, planet_name, lagna_sign):
        if planet_name not in cls.OWN_SIGNS:
            return []
        lagna_idx = AstroMath.ZODIAC_SIGNS.index(lagna_sign)
        ruled = []
        for sign in cls.OWN_SIGNS[planet_name]:
            sign_idx = AstroMath.ZODIAC_SIGNS.index(sign)
            h = (sign_idx - lagna_idx) % 12 + 1
            ruled.append(h)
        return ruled

    @classmethod
    def get_lord_of_house(cls, house_num, lagna_sign):
        lagna_idx = AstroMath.ZODIAC_SIGNS.index(lagna_sign)
        sign_idx = (lagna_idx + house_num - 1) % 12
        sign_name = AstroMath.ZODIAC_SIGNS[sign_idx]
        return cls.LORDSHIPS[sign_name]

    @classmethod
    def get_aspecting_planets(cls, target_house, chart, is_sid):
        aspecting = []
        for name, p in chart.items():
            if name in ["Rahu", "Ketu"]:
                continue
            p_house = p.vedic_house if is_sid else p.tropical_house
            d = (target_house - p_house) % 12 + 1
            has_aspect = False
            if d == 7:
                has_aspect = True
            elif name == "Mars" and d in [4, 8]:
                has_aspect = True
            elif name == "Jupiter" and d in [5, 9]:
                has_aspect = True
            elif name == "Saturn" and d in [3, 10]:
                has_aspect = True
            if has_aspect:
                aspecting.append(name)
        return aspecting

    @classmethod
    def resolve_candidates(cls, expr, chart, lagna_sign, is_sid, bindings):
        if not expr:
            return []
        if isinstance(expr, list):
            res = []
            for item in expr:
                res.extend(cls.resolve_candidates(item, chart, lagna_sign, is_sid, bindings))
            return sorted(list(set(res)))
            
        if expr in bindings:
            return [bindings[expr]]
        if expr in chart:
            return [expr]
        if expr == "Lagna":
            return ["Lagna"]
        if expr == "Chart":
            return ["Chart"]
        if expr in ["LagnaLord", "LordOf(1)"]:
            return [cls.get_lord_of_house(1, lagna_sign)]
            
        if expr.startswith("LordOf("):
            try:
                house = int(expr[7:-1])
                return [cls.get_lord_of_house(house, lagna_sign)]
            except:
                pass
                
        if expr.startswith("DispositorOf("):
            inner = expr[13:-1]
            resolved = cls.resolve_candidates(inner, chart, lagna_sign, is_sid, bindings)
            if resolved:
                p_name = resolved[0]
                if p_name in chart:
                    lon = chart[p_name].sidereal_longitude if is_sid else chart[p_name].tropical_longitude
                    sign = AstroMath.get_zodiac(lon)
                    return [cls.LORDSHIPS[sign]]
            return []
            
        if expr.startswith("NavamsaDispositorOf("):
            inner = expr[20:-1]
            resolved = cls.resolve_candidates(inner, chart, lagna_sign, is_sid, bindings)
            if resolved:
                p_name = resolved[0]
                if p_name in chart:
                    vargas = chart[p_name].vargas
                    d9_sign = vargas.get("D9")
                    if d9_sign:
                        return [cls.LORDSHIPS[d9_sign]]
            return []
            
        if expr.startswith("ExaltationLordOf("):
            inner = expr[17:-1]
            resolved = cls.resolve_candidates(inner, chart, lagna_sign, is_sid, bindings)
            if resolved:
                p_name = resolved[0]
                exalt_sign = cls.EXALTATION_SIGNS.get(p_name)
                if exalt_sign:
                    return [cls.LORDSHIPS[exalt_sign]]
            return []
            
        if expr in ["AnyPlanet", "AllPlanets"]:
            return [n for n in chart]
        if expr in ["AnyBeneficPlanet", "AllBeneficPlanets", "BeneficPlanet"]:
            return [n for n in ["Mercury", "Venus", "Jupiter"] if n in chart]
        if expr in ["AnyMaleficPlanet", "AllMaleficPlanets"]:
            return [n for n in ["Sun", "Mars", "Saturn", "Rahu", "Ketu"] if n in chart]
        if expr == "AnyUnblemishedBenefic":
            sun = chart.get("Sun")
            return [n for n in ["Mercury", "Venus", "Jupiter"] if n in chart and cls.is_unblemished(chart[n], sun, is_sid)]
        if expr == "AnyDebilitatedPlanet":
            return [n for n in chart if cls.get_rules_dignity(n, chart[n].sidereal_longitude if is_sid else chart[n].tropical_longitude, chart, is_sid) == "Debilitated"]
        if expr == "AnyStrongPlanet":
            return [n for n in chart if cls.is_planet_strong(n, chart[n])]
            
        return []

    @classmethod
    def get_property_value(cls, cand_name, prop_name, rule, chart, lagna_sign, is_sid, bindings, subject_profile):
        if cand_name == "Chart":
            if prop_name == "birth_time":
                sun = chart.get("Sun")
                sun_house = sun.vedic_house if is_sid else sun.tropical_house
                return "Day" if sun_house in [7, 8, 9, 10, 11, 12] else "Night"
            if prop_name == "native_gender":
                if subject_profile:
                    return subject_profile.get("gender", "Male")
                return "Male"
                
        if cand_name in chart:
            p = chart[cand_name]
            if prop_name == "house":
                return p.vedic_house if is_sid else p.tropical_house
            if prop_name == "dignity":
                lon = p.sidereal_longitude if is_sid else p.tropical_longitude
                return cls.get_rules_dignity(cand_name, lon, chart, is_sid)
            if prop_name == "relative_house_to_sun":
                sun = chart.get("Sun")
                h = p.vedic_house if is_sid else p.tropical_house
                sun_h = sun.vedic_house if is_sid else sun.tropical_house
                return (h - sun_h) % 12 + 1
            if prop_name == "relative_house_to_moon":
                moon = chart.get("Moon")
                h = p.vedic_house if is_sid else p.tropical_house
                moon_h = moon.vedic_house if is_sid else moon.tropical_house
                return (h - moon_h) % 12 + 1
            if prop_name == "relative_house_to_lagna":
                return p.vedic_house if is_sid else p.tropical_house
            if prop_name == "relative_house_to_lagna_or_moon":
                h = p.vedic_house if is_sid else p.tropical_house
                moon = chart.get("Moon")
                moon_h = moon.vedic_house if is_sid else moon.tropical_house
                rel_moon = (h - moon_h) % 12 + 1
                return 10 if (h == 10 or rel_moon == 10) else 0
            if prop_name == "relative_house_to_jupiter":
                jupiter = chart.get("Jupiter")
                h = p.vedic_house if is_sid else p.tropical_house
                jup_h = jupiter.vedic_house if is_sid else jupiter.tropical_house
                return (h - jup_h) % 12 + 1
            if prop_name == "relative_house_to_target":
                t_param = rule.get("target_parameter")
                resolved = cls.resolve_candidates(t_param, chart, lagna_sign, is_sid, bindings)
                if resolved and resolved[0] in chart:
                    h = p.vedic_house if is_sid else p.tropical_house
                    t_h = chart[resolved[0]].vedic_house if is_sid else chart[resolved[0]].tropical_house
                    return (h - t_h) % 12 + 1
                return 0
            if prop_name == "aspects_target":
                t_param = rule.get("target_parameter")
                resolved = cls.resolve_candidates(t_param, chart, lagna_sign, is_sid, bindings)
                if resolved and resolved[0] in chart:
                    p_house = p.vedic_house if is_sid else p.tropical_house
                    t_house = chart[resolved[0]].vedic_house if is_sid else chart[resolved[0]].tropical_house
                    d = (t_house - p_house) % 12 + 1
                    if d == 7:
                        return True
                    if cand_name == "Mars" and d in [4, 8]:
                        return True
                    if cand_name == "Jupiter" and d in [5, 9]:
                        return True
                    if cand_name == "Saturn" and d in [3, 10]:
                        return True
                return False
            if prop_name == "is_aspected_by":
                p_house = p.vedic_house if is_sid else p.tropical_house
                return cls.get_aspecting_planets(p_house, chart, is_sid)
            if prop_name == "is_unblemished":
                sun = chart.get("Sun")
                return cls.is_unblemished(p, sun, is_sid)
            if prop_name == "is_strong":
                return cls.is_planet_strong(cand_name, p)
            if prop_name == "sign":
                lon = p.sidereal_longitude if is_sid else p.tropical_longitude
                return AstroMath.get_zodiac(lon)
            if prop_name == "sign_polarity":
                lon = p.sidereal_longitude if is_sid else p.tropical_longitude
                sign = AstroMath.get_zodiac(lon)
                idx = AstroMath.ZODIAC_SIGNS.index(sign)
                return "Odd" if idx % 2 == 0 else "Even"
            if prop_name == "sign_quality":
                lon = p.sidereal_longitude if is_sid else p.tropical_longitude
                return cls.get_sign_quality(lon)
            if prop_name == "dispositor":
                lon = p.sidereal_longitude if is_sid else p.tropical_longitude
                sign = AstroMath.get_zodiac(lon)
                return cls.LORDSHIPS[sign]
            if prop_name == "rules_house":
                return cls.get_ruled_houses(cand_name, lagna_sign)
            if prop_name == "is_different_from":
                return cand_name
                
        return None

    @classmethod
    def evaluate_operator(cls, actual, op, expected):
        if op == "EQUALS":
            return actual == expected
        if op == "NOT_EQUALS":
            return actual != expected
        if op == "IN":
            if isinstance(expected, list):
                return actual in expected
            return actual == expected
        if op == "NOT_IN":
            if isinstance(expected, list):
                return actual not in expected
            return actual != expected
        if op == "CONTAINS":
            if isinstance(actual, (list, set)):
                return expected in actual
            if isinstance(actual, str):
                return expected in actual
            return False
        return False

    @classmethod
    def evaluate_rule(cls, rule, chart, lagna_sign, is_sid, bindings, subject_profile=None):
        log_op = rule.get("logical_operator")
        
        if log_op == "INITIALIZE_TARGET":
            target_name = rule.get("target_name")
            target_val_expr = rule.get("target_value")
            exclude = rule.get("exclude", [])
            evaluate_block = rule.get("evaluate", {})
            
            candidates = cls.resolve_candidates(target_val_expr, chart, lagna_sign, is_sid, bindings)
            candidates = [c for c in candidates if c not in exclude]
            
            if not candidates:
                return False, []
                
            any_success = False
            successful_bindings = []
            for cand in candidates:
                new_bindings = bindings.copy()
                new_bindings[target_name] = cand
                success, matched_planets = cls.evaluate_rule(evaluate_block, chart, lagna_sign, is_sid, new_bindings, subject_profile)
                if success:
                    any_success = True
                    successful_bindings.extend(matched_planets)
                    successful_bindings.append(cand)
            return any_success, sorted(list(set(successful_bindings)))
            
        elif log_op is not None:
            conds = rule.get("conditions", [])
            results = []
            all_matched = []
            for cond in conds:
                success, matched = cls.evaluate_rule(cond, chart, lagna_sign, is_sid, bindings, subject_profile)
                results.append(success)
                if success or log_op in ["AND", "ALL"]:
                    all_matched.extend(matched)
                    
            if log_op in ["AND", "ALL", "EXISTS"]:
                passed = all(results) if results else False
            elif log_op == "OR":
                passed = any(results) if results else False
            elif log_op == "NOT_EXISTS":
                passed = not any(results)
            else:
                passed = False
                
            return passed, sorted(list(set(all_matched))) if passed else []
            
        else:
            # Simple condition
            target_expr = rule.get("target")
            prop_name = rule.get("property")
            op = rule.get("operator")
            expected_val = rule.get("value")
            exclude = rule.get("exclude", [])
            
            candidates = cls.resolve_candidates(target_expr, chart, lagna_sign, is_sid, bindings)
            candidates = [c for c in candidates if c not in exclude]
            
            if not candidates:
                return False, []
                
            if op == "UNIQUE_COUNT_EQUALS":
                vals = [cls.get_property_value(c, prop_name, rule, chart, lagna_sign, is_sid, bindings, subject_profile) for c in candidates]
                passed = (len(set(vals)) == expected_val)
                return passed, candidates if passed else []
                
            is_all = target_expr in ["AllPlanets", "AllBeneficPlanets", "AllMaleficPlanets"]
            matching_candidates = []
            for cand_name in candidates:
                actual_val = cls.get_property_value(cand_name, prop_name, rule, chart, lagna_sign, is_sid, bindings, subject_profile)
                if cls.evaluate_operator(actual_val, op, expected_val):
                    matching_candidates.append(cand_name)
                    
            if is_all:
                passed = (len(matching_candidates) == len(candidates) and len(candidates) > 0)
            else:
                passed = (len(matching_candidates) > 0)
                
            return passed, matching_candidates if passed else []

    @classmethod
    def scan_yogas(cls, chart, local_sky, mode, rules_path, subject_profile=None):
        import json
        import os
        
        yogas_list = []
        if os.path.isdir(rules_path):
            for fn in sorted(os.listdir(rules_path)):
                if fn.endswith('.json'):
                    path = os.path.join(rules_path, fn)
                    with open(path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        yogas_list.extend(data.get("yogas", []))
        else:
            if not os.path.exists(rules_path):
                raise FileNotFoundError(f"Yoga rules file not found: {rules_path}")
            with open(rules_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                yogas_list.extend(data.get("yogas", []))
                
        is_sid = False if mode in ["Tropical-Vedic", "Tropical Focus"] else True
        lagna_sign = local_sky.sidereal_ascendant_sign if is_sid else local_sky.tropical_ascendant_sign
        
        yogas_output = {}
        for y in yogas_list:
            y_name = y.get("name")
            y_desc = y.get("description")
            rules_conf = y.get("rules", {})
            
            active, matching = cls.evaluate_rule(rules_conf, chart, lagna_sign, is_sid, {}, subject_profile)
            
            log_op = rules_conf.get("logical_operator")
            details_planets = [p for p in matching if p not in ["Chart", "Lagna"]]
            
            details_str = f"Matching planets: {', '.join(details_planets)}." if active and details_planets else "No matching planets."
            if not active and log_op == "NOT_EXISTS":
                details_str = "Absence conditions not met."
            elif active and log_op == "NOT_EXISTS":
                details_str = "Absence conditions met."
                
            yogas_output[y_name] = {
                "active": active,
                "description": y_desc,
                "results": y.get("results", ""),
                "details": details_str
            }
            
        return yogas_output


# --- 7. GOCHARA TRANSIT ENGINE ---
class GocharaEngine:
    AUSPICIOUS = {
        "Sun": [3, 6, 10, 11],
        "Moon": [1, 3, 6, 7, 10, 11],
        "Mars": [3, 6, 11],
        "Mercury": [2, 4, 6, 8, 10, 11],
        "Jupiter": [2, 5, 7, 9, 11],
        "Venus": [1, 2, 3, 4, 5, 8, 9, 11, 12],
        "Saturn": [3, 6, 11],
        "Rahu": [3, 6, 11],
        "Ketu": [3, 6, 11]
    }

    @classmethod
    def evaluate_transits(cls, natal_chart, transit_chart, mode):
        m_lon = natal_chart["Moon"].tropical_longitude if mode in ["Tropical-Vedic", "Tropical Focus"] else natal_chart["Moon"].sidereal_longitude
        m_sign_idx = int(m_lon // 30) % 12
        results = {}
        for name in ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn", "Rahu", "Ketu"]:
            if name not in transit_chart:
                continue
            t_p = transit_chart[name]
            t_lon = t_p.tropical_longitude if mode in ["Tropical-Vedic", "Tropical Focus"] else t_p.sidereal_longitude
            t_sign_idx = int(t_lon // 30) % 12
            h_from_moon = (t_sign_idx - m_sign_idx) % 12 + 1
            is_auspicious = h_from_moon in cls.AUSPICIOUS.get(name, [])
            results[name] = {
                "transit_sign": AstroMath.ZODIAC_SIGNS[t_sign_idx],
                "house_from_moon": h_from_moon,
                "is_auspicious": is_auspicious
            }
        return results

# --- 8. VIMSHOTTARI DASHA ENGINE ---
class VimshottariDashaEngine:
    DASHAS = [
        ("Ketu", 7), ("Venus", 20), ("Sun", 6), ("Moon", 10),
        ("Mars", 7), ("Rahu", 18), ("Jupiter", 16), ("Saturn", 19),
        ("Mercury", 17)
    ]
    
    def __init__(self, astro_time, moon_nakshatra, moon_sid_lon):
        import datetime
        
        sequence = ["Ketu", "Venus", "Sun", "Moon", "Mars", "Rahu", "Jupiter", "Saturn", "Mercury"]
        dasha_years = {name: y for name, y in self.DASHAS}
        
        nak_idx = AstroMath.NAKSHATRAS.index(moon_nakshatra)
        lord_idx = nak_idx % 9
        
        nak_span = 13.333333333333334
        offset_in_nak = moon_sid_lon % nak_span
        fraction_remaining = (nak_span - offset_in_nak) / nak_span
        
        first_lord, first_years = self.DASHAS[lord_idx]
        remaining_years = fraction_remaining * first_years
        elapsed_years = first_years - remaining_years
        
        birth_dt = datetime.datetime(astro_time.year, astro_time.month, astro_time.day)
        mahadasha_start = birth_dt - datetime.timedelta(days=int(elapsed_years * 365.25))
        
        dasha_periods = []
        current_m_start = mahadasha_start
        
        m_idx = lord_idx
        for step in range(9):
            m_lord, m_years = self.DASHAS[m_idx]
            m_duration_days = int(m_years * 365.25)
            current_m_end = current_m_start + datetime.timedelta(days=m_duration_days)
            
            antardashas = []
            current_a_start = current_m_start
            a_sequence = [sequence[(sequence.index(m_lord) + i) % 9] for i in range(9)]
            
            for a_lord in a_sequence:
                a_years = dasha_years[a_lord]
                a_duration_days = int((m_years * a_years / 120.0) * 365.25)
                current_a_end = current_a_start + datetime.timedelta(days=a_duration_days)
                
                pratyantardashas = []
                current_p_start = current_a_start
                p_sequence = [sequence[(sequence.index(a_lord) + i) % 9] for i in range(9)]
                
                for p_lord in p_sequence:
                    p_years = dasha_years[p_lord]
                    p_duration_days = int((m_years * a_years * p_years / 14400.0) * 365.25)
                    current_p_end = current_p_start + datetime.timedelta(days=p_duration_days)
                    
                    pratyantardashas.append({
                        "lord": p_lord,
                        "start": current_p_start.strftime("%Y-%m-%d"),
                        "end": current_p_end.strftime("%Y-%m-%d"),
                        "duration_days": p_duration_days
                    })
                    current_p_start = current_p_end
                
                antardashas.append({
                    "lord": a_lord,
                    "start": current_a_start.strftime("%Y-%m-%d"),
                    "end": current_a_end.strftime("%Y-%m-%d"),
                    "pratyantardashas": pratyantardashas
                })
                current_a_start = current_a_end
                
            if step == 0:
                filtered_antardashas = []
                for a in antardashas:
                    a_end = datetime.datetime.strptime(a["end"], "%Y-%m-%d")
                    if a_end > birth_dt:
                        a_start = datetime.datetime.strptime(a["start"], "%Y-%m-%d")
                        if a_start < birth_dt:
                            a["start"] = birth_dt.strftime("%Y-%m-%d")
                            
                        filtered_pratyantardashas = []
                        for p in a["pratyantardashas"]:
                            p_end = datetime.datetime.strptime(p["end"], "%Y-%m-%d")
                            if p_end > birth_dt:
                                p_start = datetime.datetime.strptime(p["start"], "%Y-%m-%d")
                                if p_start < birth_dt:
                                    p["start"] = birth_dt.strftime("%Y-%m-%d")
                                filtered_pratyantardashas.append(p)
                        a["pratyantardashas"] = filtered_pratyantardashas
                        filtered_antardashas.append(a)
                antardashas = filtered_antardashas
                display_start = birth_dt
                display_duration = remaining_years
            else:
                display_start = current_m_start
                display_duration = m_years
                
            dasha_periods.append({
                "lord": m_lord,
                "start": display_start.strftime("%Y-%m-%d"),
                "end": current_m_end.strftime("%Y-%m-%d"),
                "duration_years": round(display_duration, 2),
                "antardashas": antardashas
            })
            current_m_start = current_m_end
            m_idx = (m_idx + 1) % 9
            
        self.dasha_periods = dasha_periods

class HTMLReporter:
    def __init__(self, json_path, template_path):
        import json
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.meta = self.data.get('metadata', {})
        self.subject = self.data.get('subject', {})
        self.chart = self.data.get('astrological_chart', {})
        self.mode = self.meta.get('settings', {}).get('report_mode', 'Everything')
        
        with open(template_path, 'r', encoding='utf-8') as f:
            self.template = f.read()

    def generate_south_chart_svg(self):
        cells = [
            (0,0), (0,1), (0,2), (0,3),
            (1,3), (2,3), (3,3), (3,2),
            (3,1), (3,0), (2,0), (1,0)
        ]
        sign_names = ["Pisces", "Aries", "Taurus", "Gemini", "Cancer", "Leo", "Virgo", "Libra", "Scorpio", "Sagittarius", "Capricorn", "Aquarius"]
        
        sign_to_planets = {s: [] for s in sign_names}
        is_sid = False if self.mode in ["Tropical-Vedic", "Tropical Focus"] else True
        
        for p in self.chart.get('planets', []):
            sign_name = p.get('tropical_sign') if not is_sid else p.get('sidereal_sign')
            if sign_name in sign_to_planets:
                sign_to_planets[sign_name].append(p['name'])
                
        lagna_sign = self.chart.get('ascendant', {}).get('tropical_sign') if not is_sid else self.chart.get('ascendant', {}).get('sidereal_sign')
        
        svg = '<svg viewBox="0 0 400 400" width="100%" height="100%" xmlns="http://www.w3.org/2000/svg" aria-labelledby="south-chart-title" role="img">\n'
        svg += '  <title id="south-chart-title">South Indian Style Astrology Chart</title>\n'
        svg += '  <desc>A 4x4 square grid showing signs clockwise starting from Pisces at top-left. Displays Lagna and planetary positions.</desc>\n'
        svg += '  <rect width="400" height="400" fill="#161730" stroke="#2e305c" stroke-width="3"/>\n'
        svg += '  <rect x="100" y="100" width="200" height="200" fill="#0c0d21" stroke="#2e305c" stroke-width="2"/>\n'
        svg += f'  <text x="200" y="190" fill="#f3c63f" font-family="Cinzel" font-size="16" text-anchor="middle">{self.subject.get("name", "")}</text>\n'
        svg += '  <text x="200" y="220" fill="#a5a7cc" font-family="Outfit" font-size="12" text-anchor="middle">South Indian Chart</text>\n'
        
        p_abbr = {"Sun": "SU", "Moon": "MO", "Mars": "MA", "Mercury": "ME", "Jupiter": "JU", "Venus": "VE", "Saturn": "SA", "Rahu": "RA", "Ketu": "KE"}
        
        for idx, (r, c) in enumerate(cells):
            sign_name = sign_names[idx]
            x = c * 100
            y = r * 100
            svg += f'  <rect x="{x}" y="{y}" width="100" height="100" fill="none" stroke="#2e305c" stroke-width="2"/>\n'
            svg += f'  <text x="{x+50}" y="{y+20}" fill="#f3c63f" font-family="Cinzel" font-size="11" text-anchor="middle" font-weight="bold">{sign_name[:3].upper()}</text>\n'
            
            content = []
            if sign_name == lagna_sign:
                content.append("ASC")
            content.extend([p_abbr.get(p_name, p_name.upper()) for p_name in sign_to_planets.get(sign_name, [])])
            
            for i, item in enumerate(content):
                cy = y + 45 + (i * 15)
                fill_color = "#00f2fe" if item == "ASC" else "#f1f1f5"
                svg += f'  <text x="{x+50}" y="{cy}" fill="{fill_color}" font-family="Outfit" font-size="11" text-anchor="middle">{item}</text>\n'
                
        svg += '</svg>'
        return svg

    def generate_north_chart_svg(self):
        is_sid = False if self.mode in ["Tropical-Vedic", "Tropical Focus"] else True
        asc_sign_name = self.chart.get('ascendant', {}).get('tropical_sign') if not is_sid else self.chart.get('ascendant', {}).get('sidereal_sign')
        
        if asc_sign_name not in AstroMath.ZODIAC_SIGNS:
            asc_sign_idx = 1
        else:
            asc_sign_idx = AstroMath.ZODIAC_SIGNS.index(asc_sign_name) + 1
        
        house_to_planets = {h: [] for h in range(1, 13)}
        for p in self.chart.get('planets', []):
            h = p.get('tropical_house') if not is_sid else p.get('vedic_house')
            if h in house_to_planets:
                house_to_planets[h].append(p['name'])
                
        house_centers = {
            1:  {"sign": (200, 90),  "planets": (200, 135)},
            2:  {"sign": (120, 45),  "planets": (100, 75)},
            3:  {"sign": (45, 120),  "planets": (65, 100)},
            4:  {"sign": (90, 200),  "planets": (135, 200)},
            5:  {"sign": (45, 280),  "planets": (65, 300)},
            6:  {"sign": (120, 355), "planets": (100, 325)},
            7:  {"sign": (200, 310), "planets": (200, 265)},
            8:  {"sign": (280, 355), "planets": (300, 325)},
            9:  {"sign": (355, 280), "planets": (335, 300)},
            10: {"sign": (310, 200), "planets": (265, 200)},
            11: {"sign": (355, 120), "planets": (335, 100)},
            12: {"sign": (280, 45),  "planets": (300, 75)}
        }
        
        svg = '<svg viewBox="0 0 400 400" width="100%" height="100%" xmlns="http://www.w3.org/2000/svg" aria-labelledby="north-chart-title" role="img">\n'
        svg += '  <title id="north-chart-title">North Indian Style Astrology Chart</title>\n'
        svg += '  <desc>A diamond-based chart showing 12 houses starting counter-clockwise from House 1 in the top center diamond. Displays Lagna and planetary positions.</desc>\n'
        svg += '  <rect width="400" height="400" fill="#161730" stroke="#2e305c" stroke-width="3"/>\n'
        svg += '  <line x1="0" y1="0" x2="400" y2="400" stroke="#2e305c" stroke-width="2"/>\n'
        svg += '  <line x1="400" y1="0" x2="0" y2="400" stroke="#2e305c" stroke-width="2"/>\n'
        svg += '  <line x1="200" y1="0" x2="0" y2="200" stroke="#2e305c" stroke-width="2"/>\n'
        svg += '  <line x1="0" y1="200" x2="200" y2="400" stroke="#2e305c" stroke-width="2"/>\n'
        svg += '  <line x1="200" y1="400" x2="400" y2="200" stroke="#2e305c" stroke-width="2"/>\n'
        svg += '  <line x1="400" y1="200" x2="200" y2="0" stroke="#2e305c" stroke-width="2"/>\n'
        
        p_abbr = {"Sun": "SU", "Moon": "MO", "Mars": "MA", "Mercury": "ME", "Jupiter": "JU", "Venus": "VE", "Saturn": "SA", "Rahu": "RA", "Ketu": "KE"}
        sign_abbreviations = ["ARI", "TAU", "GEM", "CAN", "LEO", "VIR", "LIB", "SCO", "SAG", "CAP", "AQU", "PIS"]
        
        for h in range(1, 13):
            sign_num = (asc_sign_idx + h - 2) % 12 + 1
            sign_name = sign_abbreviations[sign_num - 1]
            centers = house_centers[h]
            sx, sy = centers["sign"]
            svg += f'  <text x="{sx}" y="{sy}" fill="#f3c63f" font-family="Outfit" font-size="9" text-anchor="middle" font-weight="bold">{sign_name}</text>\n'
            
            content = []
            if h == 1:
                content.append("ASC")
            content.extend([p_abbr.get(p_name, p_name.upper()) for p_name in house_to_planets.get(h, [])])
            
            px, py = centers["planets"]
            planet_str = ", ".join(content)
            fill_color = "#00f2fe" if "ASC" in content else "#f1f1f5"
            svg += f'  <text x="{px}" y="{py}" fill="{fill_color}" font-family="Outfit" font-size="12" text-anchor="middle" font-weight="600">{planet_str}</text>\n'
            
        svg += '</svg>'
        return svg

    def compile_report(self, output_path):
        import datetime
        
        html = self.template
        
        # 1. Subject info & Metadata
        html = html.replace("{{subject_name}}", self.subject.get("name", ""))
        html = html.replace("{{birth_date}}", self.subject.get("birth_data", {}).get("date", ""))
        html = html.replace("{{birth_time}}", self.subject.get("birth_data", {}).get("time", ""))
        html = html.replace("{{birth_tz}}", self.subject.get("birth_data", {}).get("timezone", ""))
        html = html.replace("{{birth_lat}}", str(self.subject.get("birth_data", {}).get("latitude", "")))
        html = html.replace("{{birth_lon}}", str(self.subject.get("birth_data", {}).get("longitude", "")))
        html = html.replace("{{report_mode}}", self.meta.get("settings", {}).get("report_mode", ""))
        html = html.replace("{{house_system}}", self.meta.get("settings", {}).get("house_system_name", ""))
        html = html.replace("{{ayanamsa}}", self.meta.get("settings", {}).get("ayanamsa_name", ""))
        
        # 2. SVGs
        html = html.replace("{{north_chart_svg}}", self.generate_north_chart_svg())
        html = html.replace("{{south_chart_svg}}", self.generate_south_chart_svg())
        
        # 3. Planet Placements
        planet_rows = ""
        for p in self.chart.get('planets', []):
            sb_tot = p.get('shadbala', {}).get('shadbala_total', '-')
            planet_rows += f"""<tr>
                <th scope="row">{p['name']}</th>
                <td>{p.get('tropical_sign', '-')}</td>
                <td>{p.get('tropical_house', '-')}</td>
                <td>{p.get('tropical_degree_in_house', '-')}°</td>
                <td>{p.get('dignity', '-')}</td>
                <td>{sb_tot}</td>
                <td>{p.get('sidereal_nakshatra', '-')}</td>
            </tr>""" + "\n"
        html = html.replace("{{planet_rows}}", planet_rows)
        
        # 4. Planetary Friendships (using English Equivalents)
        friendship_mapping = {
            "Adhi Mitra": "Great Friend",
            "Mitra": "Friend",
            "Sama": "Neutral",
            "Shatru": "Enemy",
            "Adhi Shatru": "Bitter Enemy"
        }
        friendships_grid = ""
        classical = ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn"]
        for p1 in classical:
            friendships_grid += f"<tr>\n  <th scope=\"row\">{p1}</th>\n"
            for p2 in classical:
                if p1 == p2:
                    friendships_grid += "  <td>-</td>\n"
                else:
                    rel = self.data.get('planetary_friendships', {}).get(p1, {}).get(p2, 'Sama')
                    english_rel = friendship_mapping.get(rel, rel)
                    friendships_grid += f"  <td>{english_rel}</td>\n"
            friendships_grid += "</tr>\n"
        html = html.replace("{{friendships_grid}}", friendships_grid)
        
        # 5. Vedic Yogas
        yogas_cards = ""
        for y_name, y_data in self.chart.get('yogas', {}).items():
            status_class = "active" if y_data.get("active") else "inactive"
            status_text = "Active" if y_data.get("active") else "Inactive"
            results_text = y_data.get('results', '')
            results_html = f'<p style="margin-top: 0.5rem; font-size: 0.95rem; color: var(--gold); font-weight: 500;">Interpretation: {results_text}</p>' if results_text and y_data.get("active") else ''
            
            yogas_cards += f"""<div class="yoga-card {status_class}">
                <h3 class="yoga-title">{y_name}</h3>
                <span class="yoga-status {status_class}">{status_text}</span>
                <p>{y_data.get('description', '')}</p>
                {results_html}
                <p style="margin-top: 0.5rem; font-size: 0.9rem; color: var(--text-muted); font-style: italic;">{y_data.get('details', '')}</p>
            </div>""" + "\n"
        html = html.replace("{{yogas_cards}}", yogas_cards)
        
        # 6. Vimshottari Dasha (Nested details)
        dashas_timeline = ""
        today = datetime.datetime.now().strftime("%Y-%m-%d")
        for m in self.chart.get('vimshottari_dasha', []):
            is_m_active = (m.get('start', '') <= today <= m.get('end', ''))
            active_class = " active" if is_m_active else ""
            
            # Format Antardashas
            antardashas_html = ""
            for a in m.get('antardashas', []):
                is_a_active = (a.get('start', '') <= today <= a.get('end', ''))
                a_style = "color: var(--teal);" if is_a_active else "color: var(--text-color);"
                a_open = "open" if is_a_active else ""
                
                # Format Pratyantardashas
                pratyantardashas_html = ""
                for p in a.get('pratyantardashas', []):
                    is_p_active = (p.get('start', '') <= today <= p.get('end', ''))
                    p_style = "color: var(--teal); font-weight: 600;" if is_p_active else ""
                    pratyantardashas_html += f'<p style="margin-bottom: 0.25rem; {p_style}">• {p.get("lord", "")} sub-period: {p.get("start", "")} to {p.get("end", "")}</p>\n'
                
                antardashas_html += f"""
                <details {a_open} style="margin-bottom: 0.75rem; padding-left: 0.5rem;">
                    <summary style="cursor: pointer; {a_style} font-weight: 600;">
                        {a.get("lord", "")} Antardasha ({a.get("start", "")} to {a.get("end", "")})
                    </summary>
                    <div style="padding-left: 1.25rem; margin-top: 0.5rem; border-left: 1px dotted var(--border-color);">
                        {pratyantardashas_html}
                    </div>
                </details>
                """
            
            dashas_timeline += f"""
            <div class="timeline-item{active_class}">
                <details {"open" if is_m_active else ""}>
                    <summary class="timeline-title" style="cursor: pointer; outline: none; list-style: none;">
                        <strong>{m.get("lord", "")} Mahadasha</strong>
                        <span class="timeline-dates" style="display: block; margin-top: 0.25rem;">{m.get("start", "")} to {m.get("end", "")} ({m.get("duration_years", "")} years)</span>
                    </summary>
                    <div style="padding-left: 1rem; margin-top: 1rem; border-left: 1px dashed var(--border-color);">
                        {antardashas_html}
                    </div>
                </details>
            </div>
            """
        html = html.replace("{{dashas_timeline}}", dashas_timeline)
        
        # 7. Ashtakavarga SAV
        ashtakavarga_sav = ""
        sav = self.chart.get('ashtakavarga', {}).get('sav', [])
        for idx, sign in enumerate(AstroMath.ZODIAC_SIGNS):
            points = sav[idx] if idx < len(sav) else 0
            status = "🟢 Strong" if points > 28 else ("🔴 Weak" if points < 25 else "🟡 Neutral")
            ashtakavarga_sav += f"""<tr>
                <th scope="row">{sign}</th>
                <td>{points}</td>
                <td>{status}</td>
            </tr>""" + "\n"
        html = html.replace("{{ashtakavarga_sav}}", ashtakavarga_sav)
        
        # 8. Ashtakavarga BAV
        ashtakavarga_bav = ""
        bav = self.chart.get('ashtakavarga', {}).get('bav', {})
        for idx, sign in enumerate(AstroMath.ZODIAC_SIGNS):
            ashtakavarga_bav += f"""<tr>
                <th scope="row">{sign}</th>
                <td>{bav.get('Sun', [0]*12)[idx]}</td>
                <td>{bav.get('Moon', [0]*12)[idx]}</td>
                <td>{bav.get('Mars', [0]*12)[idx]}</td>
                <td>{bav.get('Mercury', [0]*12)[idx]}</td>
                <td>{bav.get('Jupiter', [0]*12)[idx]}</td>
                <td>{bav.get('Venus', [0]*12)[idx]}</td>
                <td>{bav.get('Saturn', [0]*12)[idx]}</td>
                <td><strong>{sav[idx] if idx < len(sav) else 0}</strong></td>
            </tr>""" + "\n"
        html = html.replace("{{ashtakavarga_bav}}", ashtakavarga_bav)
        
        # 9. Transits
        transits_rows = ""
        for name, t in self.chart.get('transits', {}).items():
            status = "🟢 Auspicious" if t.get('is_auspicious') else "🔴 Challenging"
            transits_rows += f"""<tr>
                <th scope="row">{name}</th>
                <td>{t.get('transit_sign', '-')}</td>
                <td>{t.get('house_from_moon', '-')}</td>
                <td>{status}</td>
            </tr>""" + "\n"
        html = html.replace("{{transits_rows}}", transits_rows)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html)



# 🚀 Calculation & Exporter Execution Pipeline

This cell executes the calculations sequentially:
1. Computes natal houses and instantiates planetary coordinates.
2. Caches house numbers on the planet objects directly to optimize friendship and divisional friendship loops.
3. Runs the complete Shadbala strength calculations, Ashtakavarga point grids, Yoga scanner, and Gochara transit checks.
4. Exports the structured results to JSON and runs the Obsidian converter.

> [!TIP]
> **Google Colab restoration**:
> To run this notebook in Google Colab, restore the Swiss Ephemeris path setting:
> `ephe_path = '/content/ephe'`

In [ ]:
try:
    # 1. Instantiate Core Timing and House Engine based on inputs
    epoch_time = AstroTime(B_YEAR, B_MONTH, B_DAY, B_TIME, B_TZ, sid_mode=AYANAMSA)
    local_sky = HouseEngine(epoch_time, LATITUDE, LONGITUDE, house_system=HOUSE_SYSTEM)
    my_birth_chart = ChartFactory.create_chart(epoch_time, mode=REPORT_MODE)

    # 2. Build Comprehensive Planet Data List and cache house numbers
    planets_data = []
    for name, obj in my_birth_chart.items():
        # Calculate signs once
        t_h = AstroMath.get_house_placement(obj.tropical_longitude, local_sky.houses, False)
        v_h = AstroMath.get_house_placement(obj.sidereal_longitude, local_sky.houses, True)
        
        # Cache calculated house numbers directly on the instances to eliminate redundant searches
        obj.tropical_house = t_h
        obj.vedic_house = v_h

    # 3. Calculate Complete Shadbala for the 7 classical planets (skipping Rahu/Ketu)
    for name, obj in my_birth_chart.items():
        is_sid = False if REPORT_MODE in ["Tropical-Vedic", "Tropical Focus"] else True
        house_num = obj.vedic_house if is_sid else obj.tropical_house
        shadbala = ShadbalaEngine.get_complete_shadbala(name, obj, my_birth_chart, local_sky.houses, house_num, REPORT_MODE)
        obj.shadbala = shadbala

    # 4. Formulate the planets payload array
    for name, obj in my_birth_chart.items():
        if REPORT_MODE == "Tropical-Vedic":
            planets_data.append({
                'name': name,
                'tropical_sign': obj.zodiac_sign,
                'tropical_house': obj.tropical_house,
                'tropical_degree_in_house': round(AstroMath.get_degree_in_house(obj.tropical_longitude, obj.tropical_house, local_sky.houses, False), 4),
                'vargas': obj.vargas,
                'dignity': obj.dignity,
                'shadbala': obj.shadbala,
                'sidereal_nakshatra': obj.nakshatra
            })
        else:
            planets_data.append({
                'name': name,
                'tropical_sign': obj.zodiac_sign,
                'tropical_house': obj.tropical_house,
                'tropical_degree_in_house': round(AstroMath.get_degree_in_house(obj.tropical_longitude, obj.tropical_house, local_sky.houses, False), 4),
                'vargas': obj.vargas,
                'sidereal_nakshatra': obj.nakshatra,
                'sidereal_sign': obj.sidereal_sign,
                'vedic_house': obj.vedic_house,
                'vedic_degree_in_house': round(AstroMath.get_degree_in_house(obj.sidereal_longitude, obj.vedic_house, local_sky.houses, True), 4),
                'dignity': obj.dignity,
                'tropical_dignity': obj.tropical_dignity,
                'sidereal_dignity': obj.sidereal_dignity,
                'shadbala': obj.shadbala
            })

    # 5. Calculate Ashtakavarga bindu allocations
    ashtakavarga = AshtakavargaEngine.calculate_ashtakavarga(my_birth_chart, local_sky, REPORT_MODE)

    # 6. Scan Yogas
    # 6. Scan Yogas dynamically from JSON rules file
    notebook_dir = os.path.abspath('')
    yogas_rules_path = os.path.join(notebook_dir, "yogas")
    yogas = YogaEngine.scan_yogas(my_birth_chart, local_sky, REPORT_MODE, yogas_rules_path, subject_profile)

    # 7. Evaluate Gochara Transits (dynamic evaluation for current time)
    now_utc = datetime.now(timezone.utc)
    transit_time = AstroTime(now_utc.year, now_utc.month, now_utc.day, now_utc.hour + now_utc.minute/60.0, 0.0, sid_mode=AYANAMSA)
    transit_chart = ChartFactory.create_chart(transit_time, mode=REPORT_MODE)
    transits = GocharaEngine.evaluate_transits(my_birth_chart, transit_chart, REPORT_MODE)

    # 8. Assemble the unified JSON Chart Data
    if REPORT_MODE == "Tropical-Vedic":
        ascendant_data = {
            'tropical_sign': local_sky.tropical_ascendant_sign,
            'tropical_longitude': round(local_sky.tropical_ascendant, 4),
            'tropical_degree_in_house': round(AstroMath.get_degree_in_house(local_sky.tropical_ascendant, 1, local_sky.houses, False), 4)
        }
        # only tropical info for houses
        houses_data = {
            str(h): {"tropical_start": local_sky.houses[h]["tropical_start"]}
            for h in local_sky.houses
        }
    else:
        ascendant_data = {
            'tropical_sign': local_sky.tropical_ascendant_sign,
            'tropical_longitude': round(local_sky.tropical_ascendant, 4),
            'tropical_degree_in_house': round(AstroMath.get_degree_in_house(local_sky.tropical_ascendant, 1, local_sky.houses, False), 4),
            'sidereal_sign': local_sky.sidereal_ascendant_sign,
            'sidereal_longitude': round(local_sky.sidereal_ascendant, 4),
            'sidereal_degree_in_house': round(AstroMath.get_degree_in_house(local_sky.sidereal_ascendant, 1, local_sky.houses, True), 4)
        }
        houses_data = {str(h): local_sky.houses[h] for h in local_sky.houses}

    json_chart_data = {
        'ascendant': ascendant_data,
        'planets': planets_data,
        'houses': houses_data,
        'ashtakavarga': ashtakavarga,
        'yogas': yogas,
        'transits': transits,
        'vimshottari_dasha': VimshottariDashaEngine(epoch_time, my_birth_chart['Moon'].nakshatra, my_birth_chart['Moon'].sidereal_longitude).dasha_periods
    }

    # 9. Generate Planetary Compound Friendship Data
    json_friendship_data = {}
    classical = ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn"]
    for p1 in classical:
        json_friendship_data[p1] = {}
        for p2 in classical:
            if p1 == p2: continue
            is_sid = False if REPORT_MODE in ["Tropical-Vedic", "Tropical Focus"] else True
            h1 = my_birth_chart[p1].vedic_house if is_sid else my_birth_chart[p1].tropical_house
            h2 = my_birth_chart[p2].vedic_house if is_sid else my_birth_chart[p2].tropical_house
            json_friendship_data[p1][p2] = FriendshipEngine.get_panchadha_relationship(p1, p2, h1, h2)

    # 10. Assemble and Save payload JSON
    full_chart_export = {
        'metadata': metadata,
        'fingerprint': fingerprint,
        'subject': subject_profile,
        'astrological_chart': json_chart_data,
        'planetary_friendships': json_friendship_data
    }

    os.makedirs(os.path.dirname(target_json_path), exist_ok=True)
    with open(target_json_path, 'w', encoding='utf-8') as f:
        json.dump(full_chart_export, f, indent=4, ensure_ascii=False)

    # 10.5. Assemble and Save LLM-optimized compact payload JSON
    active_yogas = {
        name: {
            "description": data.get("description"),
            "results": data.get("results"),
            "details": data.get("details")
        }
        for name, data in json_chart_data["yogas"].items()
        if data.get("active")
    }

    from datetime import datetime
    today_str = datetime.now().strftime("%Y-%m-%d")
    current_dasha = {}
    for m in json_chart_data["vimshottari_dasha"]:
        if m["start"] <= today_str <= m["end"]:
            current_dasha["mahadasha"] = {"lord": m["lord"], "start": m["start"], "end": m["end"]}
            for a in m.get("antardashas", []):
                if a["start"] <= today_str <= a["end"]:
                    current_dasha["antardasha"] = {"lord": a["lord"], "start": a["start"], "end": a["end"]}
                    for p in a.get("pratyantardashas", []):
                        if p["start"] <= today_str <= p["end"]:
                            current_dasha["pratyantardasha"] = {"lord": p["lord"], "start": p["start"], "end": p["end"]}
                            break
                    break
            break

    llm_chart_data = {
        "ascendant": json_chart_data["ascendant"],
        "planets": planets_data,
        "houses": json_chart_data["houses"],
        "ashtakavarga_sav": json_chart_data["ashtakavarga"]["sav"],
        "active_yogas": active_yogas,
        "current_dasha": current_dasha,
        "transits": json_chart_data["transits"]
    }

    llm_payload = {
        "subject": subject_profile,
        "astrological_chart": llm_chart_data,
        "planetary_friendships": json_friendship_data
    }

    target_llm_json_path = os.path.join("knowledge_base", safe_subject, f"{safe_subject}_llm_payload.json")
    with open(target_llm_json_path, 'w', encoding='utf-8') as f:
        json.dump(llm_payload, f, indent=2, ensure_ascii=False)

    # 11. Generate HTML Report
    notebook_dir = os.path.abspath('')
    template_path = os.path.join(notebook_dir, "report_template.html")
    report_output_path = os.path.join(os.path.dirname(target_json_path), f"{safe_subject}_report.html")
    reporter = HTMLReporter(target_json_path, template_path)
    reporter.compile_report(report_output_path)

    print(f"✅ Calculation Pipeline Success. Data saved to: {target_json_path}")
    print(f"✅ HTML Report successfully generated at: {report_output_path}")
    print(f"Mode: {REPORT_MODE} | Divisional Data: {list(planets_data[0].get('vargas', {}).keys())}")

except Exception as e:
    print(f"❌ Execution Error: {e}")


In [ ]:
# 1. Verify Varga Content Integrity (D9, D10, D12, D60) from JSON
subject_dir = os.path.join('knowledge_base', safe_subject)

with open(target_json_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

first_planet_vargas = json_data['astrological_chart']['planets'][0].get('vargas', {})
vargas_checks = {v: (v in first_planet_vargas) for v in ['D9', 'D10', 'D12', 'D60']}

# Clean up any existing markdown files / 'charts' directory in output
charts_dir = os.path.join(subject_dir, 'charts')
if os.path.exists(charts_dir):
    import shutil
    shutil.rmtree(charts_dir)

# 2. Verify HTML Report Existence
report_file = os.path.join(subject_dir, f"{safe_subject}_report.html")
report_exists = os.path.exists(report_file) and os.path.getsize(report_file) > 0

# 3. Create Final Production Archive
final_zip_name = f"{safe_subject}_astrology_profile_final"
archive_path = shutil.make_archive(final_zip_name, 'zip', subject_dir)

print(f"--- Final Production Verification ---")
print(f"✅ Calculation Integrity Verified from JSON for: {SUBJECT_NAME}")
print(f"✅ Varga Integrity Checked: {vargas_checks}")
print(f"✅ HTML Report Generated & Verified: {report_exists}")
print(f"✅ Final Production Archive Created: {archive_path}")
print(f"\nThe engine is now production-ready with HTML dashboard output.")
